##Imports

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

##Parameters

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

CUSTOMER_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer"
PERSON_TABLE = f"{CATALOG}.{SCHEMA}.silver_person"

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gold_dim_customer"

##Read Silver Tables

In [0]:
customer_df = spark.table(CUSTOMER_TABLE)

person_df = spark.table(PERSON_TABLE)

##Build Customer Dimension Dataset

####Join Customer + Person

In [0]:
source_df = (
    customer_df.alias("c")
    .join(
        person_df.alias("p"),
        F.col("c.PersonID") == F.col("p.BusinessEntityID"),
        "left"
    )
)

####Create Business Columns

In [0]:
source_df = (
    source_df
    .select(
        F.col("c.CustomerID"),
        F.col("c.PersonID"),
        F.col("c.StoreID"),
        F.col("c.TerritoryID"),

        F.col("p.FirstName"),
        F.col("p.MiddleName"),
        F.col("p.LastName")
    )
)

In [0]:
source_df = source_df.withColumn(
    "customer_type",
    F.when(
        F.col("StoreID").isNotNull(),
        "STORE"
    ).when(
        F.col("PersonID").isNotNull(),
        "INDIVIDUAL"
    ).otherwise("UNKNOWN")
)

####Full Name

In [0]:
source_df = source_df.withColumn(
    "FullName",
    F.when(
        F.col("PersonID").isNotNull(),
        F.concat_ws(
            " ",
            F.col("FirstName"),
            F.col("MiddleName"),
            F.col("LastName")
    )
)
)

##SCD Hash

In [0]:
source_df = source_df.withColumn(
    "customer_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("CustomerID").cast("string"),
            F.col("PersonID").cast("string"),
            F.col("StoreID").cast("string"),
            F.col("TerritoryID").cast("string"),
            F.col("customer_type"),
            F.col("FirstName"),
            F.col("MiddleName"),
            F.col("LastName")
        ),
        256
    )
)

##Add SCD Columns

In [0]:
source_df = (
    source_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .withColumn(
        "is_current",
        F.lit(1)
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

##Create Target Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE}
(
    Customer_SK BIGINT GENERATED ALWAYS AS IDENTITY,

    CustomerID BIGINT,
    PersonID BIGINT,
    StoreID BIGINT,
    TerritoryID BIGINT,

    customer_type STRING,
    FirstName STRING,
    MiddleName STRING,
    LastName STRING,
    FullName STRING,

    customer_hash STRING,

    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,

    is_current INT,

    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

##Check Existing Data

In [0]:
target_exists = spark.catalog.tableExists(TARGET_TABLE)

##Initial Load

In [0]:
target_count = spark.table(TARGET_TABLE).count()

if target_count == 0:

    source_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TARGET_TABLE)

    print("Initial Customer Dimension Load Complete")

##SCD Type 2 Incremental Logic

####Load Target

In [0]:
target_df = (
    spark.table(TARGET_TABLE)
    .filter("is_current = 1")
)

####Detect Changed Records

In [0]:
changes_df = (
    source_df.alias("s")
    .join(
        target_df.alias("t"),
        "CustomerID"
    )
    .filter(
        F.col("s.customer_hash") != F.col("t.customer_hash")
    )
    .select("s.*")
)

####Expire Old Version

In [0]:
delta_target = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

(
    delta_target.alias("t")
    .merge(
        changes_df.alias("s"),
        """
        t.CustomerID=s.CustomerID
        AND t.is_current=1
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date":
                "current_timestamp()",

            "is_current":
                "0",

            "updated_at":
                "current_timestamp()"
        }
    )
    .execute()
)

####Insert New Version

In [0]:
new_versions_df = (
    changes_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .withColumn(
        "is_current",
        F.lit(1)
    )
)

new_versions_df = new_versions_df.select(
    "CustomerID",
    "PersonID",
    "StoreID",
    "TerritoryID",

    "customer_type",

    "FirstName",
    "MiddleName",
    "LastName",
    "FullName",

    "customer_hash",

    "effective_start_date",
    "effective_end_date",

    "is_current",

    "created_at",
    "updated_at"
)


In [0]:
new_versions_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)

##Insert Brand New Customers

In [0]:
target_current = (
    spark.table(TARGET_TABLE)
    .filter("is_current=1")
)

####Find New Customers

In [0]:
new_customer_df = (
    source_df.alias("s")
    .join(
        target_current.alias("t"),
        "CustomerID",
        "left_anti"
    )
)

####Add Audit Columns

In [0]:
new_customer_df = (
    new_customer_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .withColumn(
        "is_current",
        F.lit(1)
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

####Select Final Columns

In [0]:
new_customer_df = new_customer_df.select(
    "CustomerID",
    "PersonID",
    "TerritoryID",
    "customer_type",
    "FirstName",
    "MiddleName",
    "LastName",
    "FullName",
    "customer_hash",
    "effective_start_date",
    "effective_end_date",
    "is_current",
    "created_at",
    "updated_at"
)

####Insert New Customers

In [0]:
new_customer_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)

####Validation

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {TARGET_TABLE}
        WHERE is_current = 1
    """)
)

In [0]:
%sql
SELECT
    c.CustomerID,
    c.customer_type,
    COUNT(*) AS Orders

FROM workspace.final.gold_fact_sales f

JOIN workspace.final.gold_dim_customer c
    ON f.Customer_SK = c.Customer_SK

GROUP BY
    c.CustomerID,
    c.customer_type

ORDER BY Orders DESC
LIMIT 50

In [0]:
%sql
SELECT *
FROM workspace.final.gold_dim_customer
WHERE customer_type='STORE'
LIMIT 20